In [12]:
print("najla")

najla


In [14]:
# ============================================================
# STEP 1 Ã¢â‚¬â€ EXTRACT PLACE DATA FROM WIKIDATA
#
# Output columns:
# - wikidata_id
# - place_name
# - place_type
# - wikidata_description
# - coord
# - wikipedia_url
#
# NOTE:
# The bounding box is only used to collect candidate places.
# Later, your actual CT geometries should determine which
# places belong to your Toronto study area.
# ============================================================

import requests
import pandas as pd


# ------------------------------------------------------------
# Wikidata Query Service SPARQL endpoint.
# ------------------------------------------------------------
WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"


# ------------------------------------------------------------
# Query coordinate-bearing Wikidata entities inside the
# Toronto bounding area.
#
# OPTIONAL is used for:
# - place type
# - English Wikipedia URL
#
# This means an entity is not discarded just because one
# of those fields is missing.
# ------------------------------------------------------------
query = """
SELECT DISTINCT
    ?item
    ?itemLabel
    ?itemDescription
    ?type
    ?typeLabel
    ?coord
    ?article
WHERE {

    SERVICE wikibase:box {

        ?item wdt:P625 ?coord.

        bd:serviceParam
            wikibase:cornerWest
            "Point(-79.65 43.58)"^^geo:wktLiteral.

        bd:serviceParam
            wikibase:cornerEast
            "Point(-79.10 43.86)"^^geo:wktLiteral.
    }

    OPTIONAL {
        ?item wdt:P31 ?type.
    }

    OPTIONAL {
        ?article schema:about ?item ;
                 schema:isPartOf <https://en.wikipedia.org/>.
    }

    SERVICE wikibase:label {
        bd:serviceParam wikibase:language "en".
    }
}
"""


# ------------------------------------------------------------
# Identify your research script to Wikimedia.
# Replace the email if you want to use your own contact.
# ------------------------------------------------------------
headers = {
    "User-Agent": (
        "TorontoCyclistAttractivenessResearch/1.0 "
        "(MSc research project)"
    )
}


# ------------------------------------------------------------
# Send the SPARQL query.
# ------------------------------------------------------------
response = requests.get(
    WIKIDATA_ENDPOINT,
    params={
        "query": query,
        "format": "json",
    },
    headers=headers,
    timeout=120,
)


# ------------------------------------------------------------
# Stop immediately if the request failed.
# ------------------------------------------------------------
response.raise_for_status()


# ------------------------------------------------------------
# Extract the returned Wikidata bindings.
# ------------------------------------------------------------
bindings = response.json()["results"]["bindings"]


# ------------------------------------------------------------
# Convert Wikidata JSON into a clean list of dictionaries.
# ------------------------------------------------------------
rows = []

for row in bindings:

    rows.append({

        # Q-ID such as Q12345.
        "wikidata_id": (
            row["item"]["value"]
            .split("/")[-1]
        ),

        # Human-readable place name.
        "place_name": (
            row.get("itemLabel", {})
            .get("value")
        ),

        # Wikidata P31 / instance-of type.
        "place_type": (
            row.get("typeLabel", {})
            .get("value")
        ),

        # Short Wikidata description.
        "wikidata_description": (
            row.get("itemDescription", {})
            .get("value")
        ),

        # Geographic coordinates.
        "coord": (
            row.get("coord", {})
            .get("value")
        ),

        # English Wikipedia page URL if one exists.
        "wikipedia_url": (
            row.get("article", {})
            .get("value")
        ),
    })


# ------------------------------------------------------------
# Create the raw dataframe.
# ------------------------------------------------------------
places_raw = pd.DataFrame(rows)


# ------------------------------------------------------------
# Inspect the extraction.
# ------------------------------------------------------------
print("Raw rows:", len(places_raw))

print(
    "Unique Wikidata IDs:",
    places_raw["wikidata_id"].nunique()
)

print(
    "Unique names:",
    places_raw["place_name"].nunique()
)

print(
    "Unique Wikipedia URLs:",
    places_raw["wikipedia_url"].nunique()
)

Raw rows: 5563
Unique Wikidata IDs: 4181
Unique names: 4056
Unique Wikipedia URLs: 2361


In [15]:
# ============================================================
# STEP 2A Ã¢â‚¬â€ DEDUPLICATE BY WIKIDATA ID + PLACE NAME
#
# Why:
# The same Wikidata entity can occur multiple times because
# it can have several P31 / place_type values.
#
# Example:
#
# Q123 | Example Park | park
# Q123 | Example Park | urban park
# Q123 | Example Park | tourist attraction
#
# These should become ONE entity.
# ============================================================


def first_non_null(series):
    """
    Return the first non-null value in a pandas Series.

    If every value is missing, return None.
    """

    # Remove null values.
    values = series.dropna()

    # Return None if nothing usable remains.
    if len(values) == 0:
        return None

    # Otherwise keep the first available value.
    return values.iloc[0]


def combine_unique_types(series):
    """
    Combine multiple place types belonging to the same
    Wikidata entity into one unique sorted string.
    """

    # Remove null values and whitespace.
    values = (
        series
        .dropna()
        .astype(str)
        .str.strip()
    )

    # Remove empty strings.
    values = values[
        values != ""
    ]

    # Remove repeated types and sort them.
    unique_values = sorted(
        set(values)
    )

    # Return None when no type exists.
    if len(unique_values) == 0:
        return None

    # Keep all valid types for this entity.
    return " | ".join(unique_values)


# ------------------------------------------------------------
# Group using BOTH Wikidata ID and place name.
#
# This gives one row for each unique ID/name entity.
# ------------------------------------------------------------
places_id_unique = (
    places_raw
    .groupby(
        [
            "wikidata_id",
            "place_name",
        ],
        dropna=False,
        as_index=False,
    )
    .agg({

        # Preserve all unique types belonging to the entity.
        "place_type": combine_unique_types,

        # Keep one description.
        "wikidata_description": first_non_null,

        # Keep one coordinate.
        "coord": first_non_null,

        # Keep one Wikipedia URL.
        "wikipedia_url": first_non_null,
    })
)


# ------------------------------------------------------------
# Check result after first deduplication stage.
# ------------------------------------------------------------
print(
    "Rows after ID + name grouping:",
    len(places_id_unique)
)

print(
    "Unique Wikidata IDs:",
    places_id_unique["wikidata_id"].nunique()
)

Rows after ID + name grouping: 4181
Unique Wikidata IDs: 4181


In [16]:
# ============================================================
# STEP 2B Ã¢â‚¬â€ DEDUPLICATE AGAIN BY WIKIPEDIA URL
#
# For entities WITH a Wikipedia URL:
#     one URL = one semantic Wikipedia source
#
# If several rows use the same URL:
#     keep one ID
#     keep one name
#
# For entities WITHOUT a URL:
#     DO NOT group them together.
#     They must remain separate for the later fallback.
# ============================================================


# ------------------------------------------------------------
# Separate entities that have a Wikipedia page.
# ------------------------------------------------------------
places_with_url = (
    places_id_unique[
        places_id_unique["wikipedia_url"].notna()
    ]
    .copy()
)


# ------------------------------------------------------------
# Separate entities that do NOT have a Wikipedia page.
#
# These remain individual entities.
# ------------------------------------------------------------
places_without_url = (
    places_id_unique[
        places_id_unique["wikipedia_url"].isna()
    ]
    .copy()
)


# ------------------------------------------------------------
# Group entities that point to the same Wikipedia URL.
#
# As requested:
# - choose one Wikidata ID
# - choose one place name
#
# Types are combined rather than thrown away.
# ------------------------------------------------------------
places_url_unique = (
    places_with_url
    .groupby(
        "wikipedia_url",
        as_index=False,
    )
    .agg({

        # Select one representative Wikidata ID.
        "wikidata_id": "first",

        # Select one representative place name.
        "place_name": "first",

        # Preserve all useful place types.
        "place_type": combine_unique_types,

        # Keep one available Wikidata description.
        "wikidata_description": first_non_null,

        # Keep one coordinate for the selected entity.
        "coord": first_non_null,
    })
)


# ------------------------------------------------------------
# Recombine:
#
# 1. unique Wikipedia entities
# 2. entities that never had Wikipedia URLs
# ------------------------------------------------------------
places_unique = pd.concat(
    [
        places_url_unique,
        places_without_url,
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# Check final entity count before Wikipedia retrieval.
# ------------------------------------------------------------
print(
    "Rows after URL grouping:",
    len(places_unique)
)

print(
    "Unique Wikidata IDs:",
    places_unique["wikidata_id"].nunique()
)

print(
    "Unique non-null Wikipedia URLs:",
    places_unique["wikipedia_url"].nunique()
)

print(
    "Entities without Wikipedia URL:",
    places_unique["wikipedia_url"].isna().sum()
)

Rows after URL grouping: 4181
Unique Wikidata IDs: 4181
Unique non-null Wikipedia URLs: 2361
Entities without Wikipedia URL: 1820


In [17]:
# ============================================================
# STEP 3 Ã¢â‚¬â€ GET WIKIPEDIA SUMMARIES FROM THE URLs
#
# We:
# 1. extract the Wikipedia title from each URL
# 2. send titles to Wikipedia in batches
# 3. retrieve the introductory plain-text extract
# 4. map the summary back to the URL
#
# This is much more efficient than one HTTP request per page.
# ============================================================

import time
from urllib.parse import unquote


# ------------------------------------------------------------
# English Wikipedia Action API.
# ------------------------------------------------------------
WIKIPEDIA_API = (
    "https://en.wikipedia.org/w/api.php"
)


def extract_wikipedia_title(url):
    """
    Convert a Wikipedia URL into its page title.

    Example:
    https://en.wikipedia.org/wiki/High_Park

    becomes:

    High_Park
    """

    # Return None for missing URLs.
    if pd.isna(url):
        return None

    # Extract everything appearing after "/wiki/".
    title = url.split("/wiki/")[-1]

    # Decode URL-encoded characters.
    title = unquote(title)

    return title


# ------------------------------------------------------------
# Add the extracted page title to the dataframe.
# ------------------------------------------------------------
places_unique["wikipedia_title"] = (
    places_unique["wikipedia_url"]
    .apply(extract_wikipedia_title)
)


def chunks(values, size=50):
    """
    Split a list into batches.

    Wikipedia allows up to 50 normal titles in one
    Action API query, so the default batch size is 50.
    """

    for start in range(
        0,
        len(values),
        size,
    ):
        yield values[
            start:start + size
        ]


def resolve_title(title, mappings):
    """
    Follow Wikipedia title normalization and redirects.

    Example:

    Some_Page
        -> Some Page
        -> Final Page
    """

    # Prevent accidental infinite redirect loops.
    visited = set()

    current = title

    while (
        current in mappings
        and current not in visited
    ):

        visited.add(current)

        current = mappings[current]

    return current


def retrieve_wikipedia_batch(
    titles,
    max_retries=5,
):
    """
    Retrieve introductory plain-text extracts for
    a batch of Wikipedia page titles.

    Returns:
        dictionary:
            title -> summary
    """

    # --------------------------------------------------------
    # API parameters.
    #
    # prop=extracts:
    #     retrieve page extracts
    #
    # exintro=1:
    #     only retrieve introduction
    #
    # explaintext=1:
    #     return plain text instead of HTML
    #
    # redirects=1:
    #     automatically follow redirects
    # --------------------------------------------------------
    params = {
        "action": "query",
        "prop": "extracts",
        "exintro": 1,
        "explaintext": 1,
        "redirects": 1,
        "titles": "|".join(titles),
        "format": "json",
        "formatversion": 2,
    }

    # --------------------------------------------------------
    # Retry temporary failures and rate limiting.
    # --------------------------------------------------------
    for attempt in range(max_retries):

        response = requests.get(
            WIKIPEDIA_API,
            params=params,
            headers=headers,
            timeout=60,
        )

        # ----------------------------------------------------
        # Successful request.
        # ----------------------------------------------------
        if response.status_code == 200:
            break

        # ----------------------------------------------------
        # Rate limited.
        # Respect Retry-After if Wikipedia provides it.
        # ----------------------------------------------------
        if response.status_code == 429:

            retry_after = (
                response.headers
                .get("Retry-After")
            )

            # Use Retry-After when available.
            if retry_after is not None:
                wait_seconds = float(
                    retry_after
                )

            # Otherwise use exponential backoff.
            else:
                wait_seconds = (
                    2 ** attempt
                )

            print(
                "Rate limited. Waiting",
                wait_seconds,
                "seconds."
            )

            time.sleep(wait_seconds)

            continue

        # ----------------------------------------------------
        # Retry temporary server errors.
        # ----------------------------------------------------
        if response.status_code in [
            500,
            502,
            503,
            504,
        ]:

            wait_seconds = (
                2 ** attempt
            )

            time.sleep(wait_seconds)

            continue

        # ----------------------------------------------------
        # Stop for an unexpected permanent error.
        # ----------------------------------------------------
        response.raise_for_status()

    else:

        # ----------------------------------------------------
        # All retries failed.
        # ----------------------------------------------------
        return {
            title: None
            for title in titles
        }

    # --------------------------------------------------------
    # Read JSON response.
    # --------------------------------------------------------
    data = response.json()

    query_data = data.get(
        "query",
        {}
    )

    # --------------------------------------------------------
    # Wikipedia may normalize titles.
    #
    # Example:
    # High_Park -> High Park
    # --------------------------------------------------------
    mappings = {}

    for item in query_data.get(
        "normalized",
        [],
    ):

        mappings[
            item["from"]
        ] = item["to"]

    # --------------------------------------------------------
    # Wikipedia may also redirect pages.
    # --------------------------------------------------------
    for item in query_data.get(
        "redirects",
        [],
    ):

        mappings[
            item["from"]
        ] = item["to"]

    # --------------------------------------------------------
    # Store returned summaries using the final page title.
    # --------------------------------------------------------
    extract_by_title = {}

    for page in query_data.get(
        "pages",
        [],
    ):

        # Missing page.
        if page.get("missing"):
            continue

        extract_by_title[
            page["title"]
        ] = page.get(
            "extract"
        )

    # --------------------------------------------------------
    # Map each originally requested title back to its
    # normalized/redirected Wikipedia page.
    # --------------------------------------------------------
    output = {}

    for original_title in titles:

        final_title = resolve_title(
            original_title,
            mappings,
        )

        output[
            original_title
        ] = extract_by_title.get(
            final_title
        )

    return output

In [18]:
# ============================================================
# DOWNLOAD ALL UNIQUE WIKIPEDIA SUMMARIES
# ============================================================


# ------------------------------------------------------------
# We already deduplicated URLs, so each Wikipedia page
# should only be processed once.
# ------------------------------------------------------------
unique_titles = (
    places_unique["wikipedia_title"]
    .dropna()
    .drop_duplicates()
    .tolist()
)


print(
    "Wikipedia pages to retrieve:",
    len(unique_titles)
)


# ------------------------------------------------------------
# Store all retrieved summaries here.
# ------------------------------------------------------------
summary_by_title = {}


# ------------------------------------------------------------
# Divide titles into batches of 50.
# ------------------------------------------------------------
title_batches = list(
    chunks(
        unique_titles,
        size=50,
    )
)


# ------------------------------------------------------------
# Process each batch.
# ------------------------------------------------------------
for batch_number, batch in enumerate(
    title_batches,
    start=1,
):

    # Retrieve summaries for this batch.
    batch_results = (
        retrieve_wikipedia_batch(
            batch
        )
    )

    # Add batch results to the complete dictionary.
    summary_by_title.update(
        batch_results
    )

    # Print progress.
    print(
        f"Batch {batch_number}/"
        f"{len(title_batches)} completed"
    )

    # Small pause between API calls.
    time.sleep(0.5)


# ------------------------------------------------------------
# Map summaries back to the dataframe.
# ------------------------------------------------------------
places_unique["wikipedia_summary"] = (
    places_unique["wikipedia_title"]
    .map(summary_by_title)
)


# ------------------------------------------------------------
# Check Wikipedia summary coverage ONLY among entities
# that actually have a Wikipedia URL.
# ------------------------------------------------------------
has_url = (
    places_unique["wikipedia_url"]
    .notna()
)

print(
    "Entities with Wikipedia URL:",
    has_url.sum()
)

print(
    "Entities with retrieved summary:",
    places_unique.loc[
        has_url,
        "wikipedia_summary"
    ]
    .notna()
    .sum()
)

print(
    "Summary retrieval coverage:",
    places_unique.loc[
        has_url,
        "wikipedia_summary"
    ]
    .notna()
    .mean()
)

Wikipedia pages to retrieve: 2361
Batch 1/48 completed
Batch 2/48 completed
Batch 3/48 completed
Batch 4/48 completed
Batch 5/48 completed
Batch 6/48 completed
Batch 7/48 completed
Batch 8/48 completed
Batch 9/48 completed
Batch 10/48 completed
Rate limited. Waiting 42.0 seconds.
Batch 11/48 completed
Batch 12/48 completed
Batch 13/48 completed
Batch 14/48 completed
Batch 15/48 completed
Batch 16/48 completed
Batch 17/48 completed
Batch 18/48 completed
Batch 19/48 completed
Batch 20/48 completed
Rate limited. Waiting 49.0 seconds.
Batch 21/48 completed
Batch 22/48 completed
Batch 23/48 completed
Batch 24/48 completed
Batch 25/48 completed
Batch 26/48 completed
Batch 27/48 completed
Batch 28/48 completed
Batch 29/48 completed
Batch 30/48 completed
Rate limited. Waiting 49.0 seconds.
Batch 31/48 completed
Batch 32/48 completed
Batch 33/48 completed
Batch 34/48 completed
Batch 35/48 completed
Batch 36/48 completed
Batch 37/48 completed
Batch 38/48 completed
Batch 39/48 completed
Batch 40/

In [62]:
# ============================================================
# STEP 4 Ã¢â‚¬â€ SEMANTIC FALLBACK
#
# EXACT hierarchy:
#
# Wikipedia summary
#       Ã¢â€ â€œ
# Wikidata description
#       Ã¢â€ â€œ
# place_type
#       Ã¢â€ â€œ
# drop
#
# IMPORTANT:
# We use ONE source only.
#
# We do NOT concatenate:
# Wikipedia + Wikidata + type
#
# This keeps the fallback logic clean and interpretable.
# ============================================================


def clean_value(value):
    """
    Convert a value into clean text.

    Return None when:
    - value is NaN
    - value is empty
    """

    # Missing pandas value.
    if pd.isna(value):
        return None

    # Convert to string and remove outer whitespace.
    value = str(value).strip()

    # Treat empty strings as missing.
    if value == "":
        return None

    return value


def select_semantic_text(row):
    """
    Select exactly ONE semantic source.

    Priority:
    1. Wikipedia summary
    2. Wikidata description
    3. place type
    4. None -> entity will later be dropped
    """

    # --------------------------------------------------------
    # Clean each possible semantic source.
    # --------------------------------------------------------
    wikipedia_summary = clean_value(
        row["wikipedia_summary"]
    )

    wikidata_description = clean_value(
        row["wikidata_description"]
    )

    place_type = clean_value(
        row["place_type"]
    )

    # --------------------------------------------------------
    # Priority 1:
    # Wikipedia summary.
    # --------------------------------------------------------
    if wikipedia_summary is not None:

        return pd.Series({
            "semantic_text":
                wikipedia_summary,

            "semantic_source":
                "wikipedia",
        })

    # --------------------------------------------------------
    # Priority 2:
    # Wikidata description.
    # --------------------------------------------------------
    if wikidata_description is not None:

        return pd.Series({
            "semantic_text":
                wikidata_description,

            "semantic_source":
                "wikidata",
        })

    # --------------------------------------------------------
    # Priority 3:
    # place type.
    # --------------------------------------------------------
    if place_type is not None:

        return pd.Series({
            "semantic_text":
                place_type,

            "semantic_source":
                "type_only",
        })

    # --------------------------------------------------------
    # Nothing useful exists.
    # This entity will be removed.
    # --------------------------------------------------------
    return pd.Series({
        "semantic_text":
            None,

        "semantic_source":
            "none",
    })


# ------------------------------------------------------------
# Apply the hierarchy to every UNIQUE entity.
# ------------------------------------------------------------
semantic_result = (
    places_unique
    .apply(
        select_semantic_text,
        axis=1,
    )
)


# ------------------------------------------------------------
# Add selected text and its source back to dataframe.
# ------------------------------------------------------------
places_unique[
    [
        "semantic_text",
        "semantic_source",
    ]
] = semantic_result[
    [
        "semantic_text",
        "semantic_source",
    ]
]

In [63]:
# ============================================================
# CHECK FALLBACK DISTRIBUTION BEFORE DROPPING
# ============================================================


# ------------------------------------------------------------
# Count entities using each semantic source.
# ------------------------------------------------------------
print(
    places_unique[
        "semantic_source"
    ]
    .value_counts(
        dropna=False
    )
)


# ------------------------------------------------------------
# Show percentages.
# ------------------------------------------------------------
print(
    places_unique[
        "semantic_source"
    ]
    .value_counts(
        normalize=True,
        dropna=False,
    )
    .mul(100)
    .round(2)
)

semantic_source
wikidata     3064
wikipedia     961
type_only     120
none           36
Name: count, dtype: int64
semantic_source
wikidata     73.28
wikipedia    22.98
type_only     2.87
none          0.86
Name: proportion, dtype: float64


In [64]:
# ============================================================
# DROP ENTITIES WITH NO SEMANTIC INFORMATION
# ============================================================


# ------------------------------------------------------------
# Keep:
# - wikipedia
# - wikidata
# - type_only
#
# Remove:
# - none
# ------------------------------------------------------------
semantic_places = (
    places_unique[
        places_unique[
            "semantic_source"
        ] != "none"
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Final checks.
# ------------------------------------------------------------
print(
    "Entities before semantic filtering:",
    len(places_unique)
)

print(
    "Entities after semantic filtering:",
    len(semantic_places)
)

print(
    "Entities dropped:",
    len(places_unique)
    - len(semantic_places)
)


# ------------------------------------------------------------
# Final columns needed for the next stage.
#
# Keep the name and ID for audit / spatial debugging.
# They are NOT required to be passed to the LLM.
# ------------------------------------------------------------
semantic_places = semantic_places[
    [
        "wikidata_id",
        "place_name",
        "place_type",
        "wikidata_description",
        "wikipedia_url",
        "wikipedia_summary",
        "coord",
        "semantic_source",
        "semantic_text",
    ]
]


# ------------------------------------------------------------
# Inspect the resulting clean semantic dataset.
# ------------------------------------------------------------
display(
    semantic_places.head(20)
)

Entities before semantic filtering: 4181
Entities after semantic filtering: 4145
Entities dropped: 36


,wikidata_id,place_name,place_type,wikidata_description,wikipedia_url,wikipedia_summary,coord,semantic_source,semantic_text
0,Q3578367,Q3578367,high school,"public high school in Toronto, Ontario, Canada",https://en.wikipedia.org/wiki/%C3%89cole_secon...,NaN,Point(-79.3715 43.7522),wikidata,"public high school in Toronto, Ontario, Canada"
1,Q16890845,École secondaire Toronto Ouest,high school,"high school in Brockton Village, Toronto, Onta...",https://en.wikipedia.org/wiki/%C3%89cole_secon...,NaN,Point(-79.441 43.6522),wikidata,"high school in Brockton Village, Toronto, Onta..."
2,Q14875502,École secondaire catholique Monseigneur-de-Cha...,high school,NaN,https://en.wikipedia.org/wiki/%C3%89cole_secon...,NaN,Point(-79.423 43.786),type_only,high school
3,Q65077953,École secondaire catholique Père-Philippe-Lama...,school,public separate secondary school in Eglinton E...,https://en.wikipedia.org/wiki/%C3%89cole_secon...,NaN,Point(-79.240707 43.738784),wikidata,public separate secondary school in Eglinton E...
4,Q16834547,École secondaire catholique Saint-Frère-André,high school,"high school in Brockton Village, Toronto, Onta...",https://en.wikipedia.org/wiki/%C3%89cole_secon...,NaN,Point(-79.441 43.6522),wikidata,"high school in Brockton Village, Toronto, Onta..."
5,Q14875147,Etienne Brule Park,park,"park in Ontario, Canada",https://en.wikipedia.org/wiki/%C3%89tienne_Br%...,NaN,Point(-79.4944 43.6528),wikidata,"park in Ontario, Canada"
6,Q124322743,10 Armoury Street,NaN,"courthouse site in Toronto, Canada",https://en.wikipedia.org/wiki/10_Armoury_Street,"10 Armoury Street in Toronto, Ontario, Canada,...",Point(-79.3861 43.6541),wikipedia,"10 Armoury Street in Toronto, Ontario, Canada,..."
7,Q139973819,1414 Danforth Avenue,bank building | historic building,"bank building in Toronto, Ontario, Canada",https://en.wikipedia.org/wiki/1414_Danforth_Av...,1414 Danforth Avenue (2nd floor offices munici...,Point(-79.3274 43.68271),wikipedia,1414 Danforth Avenue (2nd floor offices munici...
8,Q4569309,1958 Jim Mideon 500,NASCAR pre-modern era race,auto race run in Ontario in 1958,https://en.wikipedia.org/wiki/1958_Jim_Mideon_500,The 1958 Jim Mideon 500 (known officially as 1...,Point(-79.417777777 43.631944444),wikipedia,The 1958 Jim Mideon 500 (known officially as 1...
9,Q139855956,1975 Scarborough bus-train collision,train collision,deadly Toronto bus-train collision,https://en.wikipedia.org/wiki/1975_Scarborough...,The 1975 Scarborough bus-train collision occur...,Point(-79.2543 43.718),wikipedia,The 1975 Scarborough bus-train collision occur...


In [65]:
# ============================================================
# MAP THE ALREADY-DOWNLOADED WIKIPEDIA SUMMARIES
# TO THE FINAL UNIQUE PLACE DATASET
#
# The retrieval cell above creates:
#
#     summary_by_title: wikipedia_title -> wikipedia_summary
#
# The old version of this cell used `summary_map`, but that
# dictionary was never created in this notebook. Build the
# URL-keyed map here from `summary_by_title`, then use the URL
# as the join key.
# ============================================================

if "summary_by_title" not in globals():
    raise NameError(
        "summary_by_title is not defined. Run the Wikipedia summary retrieval cell first."
    )

if "wikipedia_title" not in places_unique.columns:
    places_unique["wikipedia_title"] = (
        places_unique["wikipedia_url"]
        .apply(extract_wikipedia_title)
    )


# ------------------------------------------------------------
# Create the missing URL -> summary dictionary.
# ------------------------------------------------------------
summary_map = (
    places_unique[
        [
            "wikipedia_url",
            "wikipedia_title",
        ]
    ]
    .dropna(subset=["wikipedia_url"])
    .drop_duplicates(subset=["wikipedia_url"])
    .assign(
        wikipedia_summary=lambda df: df["wikipedia_title"].map(summary_by_title)
    )
    .set_index("wikipedia_url")["wikipedia_summary"]
    .to_dict()
)


# ------------------------------------------------------------
# Map each final unique Wikipedia URL to its downloaded summary.
# Places without a Wikipedia URL will correctly remain NaN.
# ------------------------------------------------------------
places_unique["wikipedia_summary"] = (
    places_unique["wikipedia_url"]
    .map(summary_map)
)


# ------------------------------------------------------------
# Check ONLY records that actually have a Wikipedia URL.
# ------------------------------------------------------------
has_wikipedia_url = (
    places_unique["wikipedia_url"]
    .notna()
)

print(
    "Entities with Wikipedia URL:",
    has_wikipedia_url.sum()
)

print(
    "Entities with Wikipedia summary:",
    places_unique.loc[
        has_wikipedia_url,
        "wikipedia_summary"
    ]
    .notna()
    .sum()
)

missing_summary_with_url = (
    places_unique[
        places_unique["wikipedia_url"].notna()
        &
        places_unique["wikipedia_summary"].isna()
    ]
)

print(
    "URL exists but summary missing:",
    len(missing_summary_with_url)
)


Entities with Wikipedia URL: 2361
Entities with Wikipedia summary: 961
URL exists but summary missing: 1400


In [66]:
# ============================================================
# INSPECT SOME PLACES AFTER THE SUMMARY MAPPING
# ============================================================

# ------------------------------------------------------------
# Show only records that have Wikipedia pages so we can verify
# that the summary is actually populated.
# ------------------------------------------------------------
display(
    places_unique.loc[
        places_unique["wikipedia_url"].notna(),
        [
            "wikidata_id",
            "place_name",
            "place_type",
            "wikidata_description",
            "wikipedia_url",
            "wikipedia_summary",
        ]
    ]
    .head(20)
)

,wikidata_id,place_name,place_type,wikidata_description,wikipedia_url,wikipedia_summary
0,Q8078053,École élémentaire Pierre-Elliott-Trudeau,NaN,NaN,https://en.wikipedia.org/wiki/%C3%89cole_%C3%A...,NaN
1,Q3578367,Q3578367,high school,"public high school in Toronto, Ontario, Canada",https://en.wikipedia.org/wiki/%C3%89cole_secon...,NaN
2,Q16890845,École secondaire Toronto Ouest,high school,"high school in Brockton Village, Toronto, Onta...",https://en.wikipedia.org/wiki/%C3%89cole_secon...,NaN
3,Q14875502,École secondaire catholique Monseigneur-de-Cha...,high school,NaN,https://en.wikipedia.org/wiki/%C3%89cole_secon...,NaN
4,Q65077953,École secondaire catholique Père-Philippe-Lama...,school,public separate secondary school in Eglinton E...,https://en.wikipedia.org/wiki/%C3%89cole_secon...,NaN
5,Q16834547,École secondaire catholique Saint-Frère-André,high school,"high school in Brockton Village, Toronto, Onta...",https://en.wikipedia.org/wiki/%C3%89cole_secon...,NaN
6,Q14875147,Etienne Brule Park,park,"park in Ontario, Canada",https://en.wikipedia.org/wiki/%C3%89tienne_Br%...,NaN
7,Q124322743,10 Armoury Street,NaN,"courthouse site in Toronto, Canada",https://en.wikipedia.org/wiki/10_Armoury_Street,"10 Armoury Street in Toronto, Ontario, Canada,..."
8,Q4549087,1331 Yonge Street,NaN,NaN,https://en.wikipedia.org/wiki/1331_Yonge_Street,NaN
9,Q139973819,1414 Danforth Avenue,bank building | historic building,"bank building in Toronto, Ontario, Canada",https://en.wikipedia.org/wiki/1414_Danforth_Av...,1414 Danforth Avenue (2nd floor offices munici...


In [67]:
# ============================================================
# STEP 4 Ã¢â‚¬â€ SELECT THE FINAL SEMANTIC TEXT
#
# EXACT fallback hierarchy:
#
# 1. Wikipedia summary
# 2. Wikidata description
# 3. place_type
# 4. Drop entity
#
# Only ONE source is selected for each entity.
# ============================================================


def clean_semantic_value(value):
    """
    Convert a semantic field into clean text.

    Returns None when the value is:
    - NaN
    - empty
    - whitespace only
    """

    # --------------------------------------------------------
    # Missing pandas value.
    # --------------------------------------------------------
    if pd.isna(value):
        return None

    # --------------------------------------------------------
    # Convert to string and remove surrounding whitespace.
    # --------------------------------------------------------
    value = str(value).strip()

    # --------------------------------------------------------
    # Treat an empty string as missing.
    # --------------------------------------------------------
    if value == "":
        return None

    return value


def select_semantic_text(row):
    """
    Select ONE semantic representation using this hierarchy:

        Wikipedia summary
             Ã¢â€ â€œ
        Wikidata description
             Ã¢â€ â€œ
        place_type
             Ã¢â€ â€œ
        None
    """

    # --------------------------------------------------------
    # Read and clean each candidate source.
    # --------------------------------------------------------
    wikipedia_summary = clean_semantic_value(
        row["wikipedia_summary"]
    )

    wikidata_description = clean_semantic_value(
        row["wikidata_description"]
    )

    place_type = clean_semantic_value(
        row["place_type"]
    )


    # --------------------------------------------------------
    # Priority 1 Ã¢â‚¬â€ Wikipedia summary.
    # --------------------------------------------------------
    if wikipedia_summary is not None:

        return pd.Series({
            "semantic_text": wikipedia_summary,
            "semantic_source": "wikipedia",
        })


    # --------------------------------------------------------
    # Priority 2 Ã¢â‚¬â€ Wikidata description.
    # --------------------------------------------------------
    if wikidata_description is not None:

        return pd.Series({
            "semantic_text": wikidata_description,
            "semantic_source": "wikidata",
        })


    # --------------------------------------------------------
    # Priority 3 Ã¢â‚¬â€ Place type.
    # --------------------------------------------------------
    if place_type is not None:

        return pd.Series({
            "semantic_text": place_type,
            "semantic_source": "type_only",
        })


    # --------------------------------------------------------
    # No useful semantic information is available.
    # This entity will be dropped.
    # --------------------------------------------------------
    return pd.Series({
        "semantic_text": None,
        "semantic_source": "none",
    })


# ------------------------------------------------------------
# Apply the fallback to the FINAL UNIQUE entity dataframe.
# ------------------------------------------------------------
semantic_selection = (
    places_unique
    .apply(
        select_semantic_text,
        axis=1
    )
)


# ------------------------------------------------------------
# Attach the resulting fields.
# ------------------------------------------------------------
places_unique[
    [
        "semantic_text",
        "semantic_source",
    ]
] = semantic_selection[
    [
        "semantic_text",
        "semantic_source",
    ]
]

In [68]:
# ============================================================
# VERIFY SEMANTIC SOURCE DISTRIBUTION
# ============================================================

# ------------------------------------------------------------
# Count how many unique entities use each semantic source.
# ------------------------------------------------------------
print(
    places_unique["semantic_source"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# Calculate percentages.
# ------------------------------------------------------------
print(
    places_unique["semantic_source"]
    .value_counts(
        normalize=True,
        dropna=False
    )
    .mul(100)
    .round(2)
)

semantic_source
wikidata     3064
wikipedia     961
type_only     120
none           36
Name: count, dtype: int64
semantic_source
wikidata     73.28
wikipedia    22.98
type_only     2.87
none          0.86
Name: proportion, dtype: float64


In [69]:
# ============================================================
# DROP ONLY ENTITIES WITH NO USABLE SEMANTIC INFORMATION
# ============================================================

# ------------------------------------------------------------
# Keep Wikipedia, Wikidata and type-only records.
#
# Drop ONLY semantic_source == "none".
# ------------------------------------------------------------
semantic_places = (
    places_unique[
        places_unique["semantic_source"] != "none"
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Report final numbers.
# ------------------------------------------------------------
print(
    "Entities before semantic filtering:",
    len(places_unique)
)

print(
    "Entities after semantic filtering:",
    len(semantic_places)
)

print(
    "Entities dropped:",
    len(places_unique) - len(semantic_places)
)

Entities before semantic filtering: 4181
Entities after semantic filtering: 4145
Entities dropped: 36


In [70]:
# ============================================================
# Cell 1 Ã¢â‚¬â€ Clean and parse Wikidata coordinates
#
# Purpose:
# Convert the Wikidata coordinate column:
#
#     Point(-79.3832 43.6532)
#
# into:
#
#     attr_lon = -79.3832
#     attr_lat =  43.6532
#     geometry = Shapely Point
#
# Invalid or missing coordinates are removed because they
# cannot be spatially joined to Census Tract polygons.
#
# Input:
#     semantic_places
#
# Required column:
#     coord
#
# Output:
#     attractions
#
# New columns:
#     attr_lon
#     attr_lat
#     geometry
# ============================================================

import pandas as pd
import geopandas as gpd
from shapely import wkt


# ------------------------------------------------------------
# Make a copy so the original semantic_places dataframe
# remains unchanged.
# ------------------------------------------------------------
attractions = semantic_places.copy()


# ------------------------------------------------------------
# Function to safely parse Wikidata WKT coordinates.
#
# Example:
#     "Point(-79.3832 43.6532)"
#
# becomes a Shapely Point object.
# ------------------------------------------------------------
def parse_wikidata_coord(value):
    """
    Parse a Wikidata WKT coordinate into a Shapely Point.

    Returns None when:
    - the coordinate is missing
    - the coordinate cannot be parsed
    - the parsed geometry is not a Point
    """

    # Handle null / NaN coordinate values.
    if pd.isna(value):
        return None

    try:
        # Convert the text into a Shapely geometry.
        geom = wkt.loads(str(value).strip())

        # We only expect Point geometry from Wikidata P625.
        if geom.geom_type != "Point":
            return None

        return geom

    except Exception:
        # Any malformed coordinate is treated as unusable.
        return None


# ------------------------------------------------------------
# Parse the original Wikidata coord column.
# ------------------------------------------------------------
attractions["geometry"] = (
    attractions["coord"]
    .apply(parse_wikidata_coord)
)


# ------------------------------------------------------------
# Report how many coordinates failed before removing them.
# ------------------------------------------------------------
invalid_coord_count = attractions["geometry"].isna().sum()

print(
    "Rows before coordinate cleaning:",
    len(attractions)
)

print(
    "Missing / invalid coordinates:",
    invalid_coord_count
)


# ------------------------------------------------------------
# Remove attractions that cannot be spatially located.
# ------------------------------------------------------------
attractions = (
    attractions
    .dropna(subset=["geometry"])
    .copy()
)


# ------------------------------------------------------------
# Wikidata coordinates are longitude / latitude.
#
# Point(x, y):
#     x = longitude
#     y = latitude
# ------------------------------------------------------------
attractions["attr_lon"] = (
    attractions["geometry"]
    .apply(lambda geom: geom.x)
)

attractions["attr_lat"] = (
    attractions["geometry"]
    .apply(lambda geom: geom.y)
)


# ------------------------------------------------------------
# Validate normal geographic coordinate ranges.
#
# Longitude must be between -180 and 180.
# Latitude must be between -90 and 90.
# ------------------------------------------------------------
valid_coordinate_range = (
    attractions["attr_lon"].between(-180, 180)
    &
    attractions["attr_lat"].between(-90, 90)
)


# ------------------------------------------------------------
# Count any coordinates outside valid geographic ranges.
# ------------------------------------------------------------
invalid_range_count = (
    ~valid_coordinate_range
).sum()

print(
    "Coordinates outside valid lon/lat range:",
    invalid_range_count
)


# ------------------------------------------------------------
# Remove coordinates outside valid geographic ranges.
# ------------------------------------------------------------
attractions = (
    attractions.loc[
        valid_coordinate_range
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Convert the dataframe into a GeoDataFrame.
#
# Wikidata P625 coordinates use geographic longitude/latitude,
# so EPSG:4326 is assigned here.
# ------------------------------------------------------------
attractions = gpd.GeoDataFrame(
    attractions,
    geometry="geometry",
    crs="EPSG:4326",
)


# ------------------------------------------------------------
# Final validation.
# ------------------------------------------------------------
print(
    "\nRows after coordinate cleaning:",
    len(attractions)
)

print(
    "Missing longitude:",
    attractions["attr_lon"].isna().sum()
)

print(
    "Missing latitude:",
    attractions["attr_lat"].isna().sum()
)

print(
    "CRS:",
    attractions.crs
)


# ------------------------------------------------------------
# Inspect the cleaned coordinates together with the attraction
# identity and semantic source.
#
# place_name is retained only for auditing/debugging.
# It is not part of semantic_text.
# ------------------------------------------------------------
display(
    attractions[
        [
            "wikidata_id",
            "place_name",
            "coord",
            "attr_lon",
            "attr_lat",
            "semantic_source",
            "semantic_text",
            "geometry",
        ]
    ]
    .head(10)
)

Rows before coordinate cleaning: 4145
Missing / invalid coordinates: 0
Coordinates outside valid lon/lat range: 0

Rows after coordinate cleaning: 4145
Missing longitude: 0
Missing latitude: 0
CRS: EPSG:4326


,wikidata_id,place_name,coord,attr_lon,attr_lat,semantic_source,semantic_text,geometry
0,Q3578367,Q3578367,Point(-79.3715 43.7522),-79.371500,43.752200,wikidata,"public high school in Toronto, Ontario, Canada",POINT (-79.3715 43.7522)
1,Q16890845,École secondaire Toronto Ouest,Point(-79.441 43.6522),-79.441000,43.652200,wikidata,"high school in Brockton Village, Toronto, Onta...",POINT (-79.441 43.6522)
2,Q14875502,École secondaire catholique Monseigneur-de-Cha...,Point(-79.423 43.786),-79.423000,43.786000,type_only,high school,POINT (-79.423 43.786)
3,Q65077953,École secondaire catholique Père-Philippe-Lama...,Point(-79.240707 43.738784),-79.240707,43.738784,wikidata,public separate secondary school in Eglinton E...,POINT (-79.24071 43.73878)
4,Q16834547,École secondaire catholique Saint-Frère-André,Point(-79.441 43.6522),-79.441000,43.652200,wikidata,"high school in Brockton Village, Toronto, Onta...",POINT (-79.441 43.6522)
5,Q14875147,Etienne Brule Park,Point(-79.4944 43.6528),-79.494400,43.652800,wikidata,"park in Ontario, Canada",POINT (-79.4944 43.6528)
6,Q124322743,10 Armoury Street,Point(-79.3861 43.6541),-79.386100,43.654100,wikipedia,"10 Armoury Street in Toronto, Ontario, Canada,...",POINT (-79.3861 43.6541)
7,Q139973819,1414 Danforth Avenue,Point(-79.3274 43.68271),-79.327400,43.682710,wikipedia,1414 Danforth Avenue (2nd floor offices munici...,POINT (-79.3274 43.68271)
8,Q4569309,1958 Jim Mideon 500,Point(-79.417777777 43.631944444),-79.417778,43.631944,wikipedia,The 1958 Jim Mideon 500 (known officially as 1...,POINT (-79.41778 43.63194)
9,Q139855956,1975 Scarborough bus-train collision,Point(-79.2543 43.718),-79.254300,43.718000,wikipedia,The 1975 Scarborough bus-train collision occur...,POINT (-79.2543 43.718)


In [71]:
# ============================================================
# Cell 2 Ã¢â‚¬â€ Spatially assign semantic attractions to CT nodes
#
# Purpose:
# 1. Load the graph nodes and CT polygon geometries.
# 2. Keep only polygons corresponding to graph nodes.
# 3. Reproject the cleaned semantic attractions to the same CRS.
# 4. Spatially join every attraction to its containing CT.
# 5. Attach the corresponding graph-node coordinates.
#
# Input:
#     attractions
#         Clean semantic attraction GeoDataFrame from Cell 1.
#
# Output:
#     attractions_with_nodes
#
# Important resulting columns:
#     wikidata_id
#     place_name
#     semantic_text
#     semantic_source
#     attr_lon
#     attr_lat
#     loc_id
#     loc_name
#     node_lon
#     node_lat
#     geometry
# ============================================================

from pathlib import Path
import geopandas as gpd
import pandas as pd


# ------------------------------------------------------------
# Define project paths.
# ------------------------------------------------------------
project_root = Path(
    "/home/najla/dev/najla-msc/bikeshare"
)

graph_dir = (
    project_root
    / "data/processed/graph"
)


# ------------------------------------------------------------
# Load graph nodes.
#
# reset_index() is retained from your existing graph pipeline.
# ------------------------------------------------------------
nodes = (
    gpd.read_parquet(
        graph_dir / "nodes.parquet"
    )
    .reset_index()
)


# ------------------------------------------------------------
# Load Census Tract polygons.
#
# Keep only:
# - loc_id
# - loc_name
# - polygon geometry
# ------------------------------------------------------------
node_polys = (
    gpd.read_parquet(
        graph_dir / "locations_gdf.parquet"
    )[
        [
            "loc_id",
            "loc_name",
            "geometry",
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# locations_gdf contains one additional loc_id = "NA".
#
# Keep only polygons that correspond to actual graph nodes.
# ------------------------------------------------------------
node_polys = (
    node_polys[
        node_polys["loc_id"]
        .isin(nodes["loc_id"])
    ]
    .copy()
)


# ------------------------------------------------------------
# Basic validation of the graph data.
# ------------------------------------------------------------
print(
    "Graph nodes:",
    len(nodes)
)

print(
    "Graph CT polygons:",
    len(node_polys)
)

print(
    "Node polygon CRS:",
    node_polys.crs
)

print(
    "Attraction CRS before reprojection:",
    attractions.crs
)


# ------------------------------------------------------------
# Reproject the attraction POINT geometries to the same CRS
# as the CT polygons.
#
# IMPORTANT:
# attr_lon and attr_lat are already stored separately in
# EPSG:4326 from Cell 1.
#
# Therefore those original longitude/latitude values are
# preserved even if the geometry itself is projected here.
# ------------------------------------------------------------
attractions_projected = (
    attractions
    .to_crs(node_polys.crs)
    .copy()
)


# ------------------------------------------------------------
# Spatially join each attraction point to the CT polygon
# containing it.
#
# how="left":
#     Keep every attraction, including points that fail to
#     fall inside one of the graph CT polygons.
#
# predicate="within":
#     The attraction point must lie within the CT polygon.
# ------------------------------------------------------------
attractions_with_loc = (
    gpd.sjoin(
        attractions_projected,
        node_polys,
        how="left",
        predicate="within",
    )
    .drop(
        columns=["index_right"],
        errors="ignore",
    )
)


# ------------------------------------------------------------
# Prepare graph-node point information.
#
# Rename the fields so attraction coordinates and node
# coordinates remain clearly distinguishable.
# ------------------------------------------------------------
node_points = (
    nodes[
        [
            "loc_id",
            "lon",
            "lat",
            "geometry",
        ]
    ]
    .rename(
        columns={
            "lon": "node_lon",
            "lat": "node_lat",
            "geometry": "node_point_geom",
        }
    )
    .copy()
)


# ------------------------------------------------------------
# Attach the corresponding graph-node information to every
# attraction using loc_id.
#
# Attractions without a CT assignment will keep NaN node
# information because this is a left merge.
# ------------------------------------------------------------
attractions_with_nodes = (
    attractions_with_loc
    .merge(
        node_points,
        on="loc_id",
        how="left",
    )
)


# ============================================================
# VALIDATION
# ============================================================


# ------------------------------------------------------------
# Count attractions successfully assigned to a graph CT.
# ------------------------------------------------------------
n_assigned = (
    attractions_with_nodes["loc_id"]
    .notna()
    .sum()
)


# ------------------------------------------------------------
# Count attractions not assigned to any graph CT.
#
# These may lie inside the initial Toronto bounding box but
# outside the actual graph/study-area polygons.
# ------------------------------------------------------------
n_unassigned = (
    attractions_with_nodes["loc_id"]
    .isna()
    .sum()
)


# ------------------------------------------------------------
# Count how many graph CTs contain at least one attraction.
# ------------------------------------------------------------
n_ct_with_attractions = (
    attractions_with_nodes["loc_id"]
    .nunique()
)


# ------------------------------------------------------------
# Number of graph CTs with zero assigned attractions.
# ------------------------------------------------------------
n_ct_without_attractions = (
    nodes["loc_id"].nunique()
    - n_ct_with_attractions
)


print(
    "\nAttractions before spatial join:",
    len(attractions)
)

print(
    "Attractions assigned to graph CTs:",
    n_assigned
)

print(
    "Attractions outside graph CTs:",
    n_unassigned
)

print(
    "\nTotal graph CTs:",
    nodes["loc_id"].nunique()
)

print(
    "CTs with >= 1 attraction:",
    n_ct_with_attractions
)

print(
    "CTs with 0 attractions:",
    n_ct_without_attractions
)


# ------------------------------------------------------------
# Inspect the spatially joined data.
# ------------------------------------------------------------
display(
    attractions_with_nodes[
        [
            "wikidata_id",
            "place_name",
            "semantic_source",
            "semantic_text",
            "attr_lon",
            "attr_lat",
            "loc_id",
            "loc_name",
            "node_lon",
            "node_lat",
        ]
    ]
    .head(20)
)

Graph nodes: 1248
Graph CT polygons: 1248
Node polygon CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "d

,wikidata_id,place_name,semantic_source,semantic_text,attr_lon,attr_lat,loc_id,loc_name,node_lon,node_lat
0,Q3578367,Q3578367,wikidata,"public high school in Toronto, Ontario, Canada",-79.371500,43.752200,5350273.02,0273.02,-79.373296,43.753904
1,Q16890845,École secondaire Toronto Ouest,wikidata,"high school in Brockton Village, Toronto, Onta...",-79.441000,43.652200,5350053.00,0053.00,-79.441967,43.654413
2,Q14875502,École secondaire catholique Monseigneur-de-Cha...,type_only,high school,-79.423000,43.786000,5350319.00,0319.00,-79.427736,43.788277
3,Q65077953,École secondaire catholique Père-Philippe-Lama...,wikidata,public separate secondary school in Eglinton E...,-79.240707,43.738784,5350355.04,0355.04,-79.242831,43.736569
4,Q16834547,École secondaire catholique Saint-Frère-André,wikidata,"high school in Brockton Village, Toronto, Onta...",-79.441000,43.652200,5350053.00,0053.00,-79.441967,43.654413
5,Q14875147,Etienne Brule Park,wikidata,"park in Ontario, Canada",-79.494400,43.652800,5350150.00,0150.00,-79.492909,43.654389
6,Q124322743,10 Armoury Street,wikipedia,"10 Armoury Street in Toronto, Ontario, Canada,...",-79.386100,43.654100,5350035.00,0035.00,-79.384952,43.656109
7,Q139973819,1414 Danforth Avenue,wikipedia,1414 Danforth Avenue (2nd floor offices munici...,-79.327400,43.682710,5350082.00,0082.00,-79.328752,43.684222
8,Q4569309,1958 Jim Mideon 500,wikipedia,The 1958 Jim Mideon 500 (known officially as 1...,-79.417778,43.631944,5350008.02,0008.02,-79.412563,43.633826
9,Q139855956,1975 Scarborough bus-train collision,wikipedia,The 1975 Scarborough bus-train collision occur...,-79.254300,43.718000,5350343.00,0343.00,-79.263601,43.711094


In [72]:
# ============================================================
# Cell 3 Ã¢â‚¬â€ Prepare node-level semantic coverage
#
# Purpose:
# 1. Remove attractions that were not assigned to a graph CT.
# 2. Verify that the spatial join did not duplicate attractions.
# 3. Count semantic attractions per CT.
# 4. Add ALL graph nodes, including CTs with zero attractions.
# 5. Create semantic_missing:
#
#       CT with attractions:
#           semantic_attraction_count > 0
#           semantic_missing = 0
#
#       CT without attractions:
#           semantic_attraction_count = 0
#           semantic_missing = 1
#
# NOTE:
# We are NOT creating embeddings yet.
# ============================================================


# ------------------------------------------------------------
# Keep only attractions successfully assigned to a graph CT.
#
# The 51 attractions outside the graph CT polygons are removed
# because they do not belong to any node in the graph.
# ------------------------------------------------------------
attractions_assigned = (
    attractions_with_nodes[
        attractions_with_nodes["loc_id"].notna()
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Validate the number of retained attractions.
# Expected from Cell 2:
#     4119
# ------------------------------------------------------------
print(
    "Attractions retained after spatial filtering:",
    len(attractions_assigned)
)


# ------------------------------------------------------------
# Check whether one attraction was accidentally assigned to
# more than one CT.
#
# Because Wikidata entities were already deduplicated earlier,
# each wikidata_id should normally appear once here.
# ------------------------------------------------------------
duplicated_attraction_ids = (
    attractions_assigned["wikidata_id"]
    .duplicated()
    .sum()
)

print(
    "Duplicated attraction IDs after spatial join:",
    duplicated_attraction_ids
)


# ------------------------------------------------------------
# If duplicates exist, inspect them before proceeding.
#
# This could indicate overlapping polygons or some unexpected
# duplication introduced during the join.
# ------------------------------------------------------------
if duplicated_attraction_ids > 0:

    duplicated_ids = (
        attractions_assigned.loc[
            attractions_assigned["wikidata_id"]
            .duplicated(keep=False),
            "wikidata_id"
        ]
        .unique()
    )

    display(
        attractions_assigned[
            attractions_assigned["wikidata_id"]
            .isin(duplicated_ids)
        ][
            [
                "wikidata_id",
                "place_name",
                "loc_id",
                "loc_name",
                "semantic_text",
            ]
        ]
        .sort_values(
            [
                "wikidata_id",
                "loc_id",
            ]
        )
    )


# ------------------------------------------------------------
# Count how many semantic attractions belong to each CT.
#
# Result example:
#
# loc_id       semantic_attraction_count
# 5350001.00   4
# 5350002.00   11
# 5350003.00   1
# ------------------------------------------------------------
ct_attraction_counts = (
    attractions_assigned
    .groupby("loc_id")
    .size()
    .reset_index(
        name="semantic_attraction_count"
    )
)


# ------------------------------------------------------------
# Start from ALL graph nodes.
#
# This is important because approximately half of your CTs
# currently have no Wikidata/Wikipedia semantic attractions.
# ------------------------------------------------------------
node_semantic_status = (
    nodes[
        [
            "loc_id",
            "loc_name",
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# Join attraction counts onto all graph nodes.
#
# Nodes without attractions receive NaN temporarily.
# ------------------------------------------------------------
node_semantic_status = (
    node_semantic_status
    .merge(
        ct_attraction_counts,
        on="loc_id",
        how="left",
    )
)


# ------------------------------------------------------------
# A missing attraction count after the merge means that the
# CT genuinely has zero retained semantic attractions.
#
# Use 0 rather than -1 because this is a real count variable.
# ------------------------------------------------------------
node_semantic_status[
    "semantic_attraction_count"
] = (
    node_semantic_status[
        "semantic_attraction_count"
    ]
    .fillna(0)
    .astype(int)
)


# ------------------------------------------------------------
# Create an explicit semantic missingness indicator.
#
# semantic_missing = 0
#     at least one semantic attraction exists
#
# semantic_missing = 1
#     no semantic attraction exists
# ------------------------------------------------------------
node_semantic_status[
    "semantic_missing"
] = (
    node_semantic_status[
        "semantic_attraction_count"
    ]
    .eq(0)
    .astype(int)
)


# ============================================================
# VALIDATION
# ============================================================


# ------------------------------------------------------------
# Number of nodes with semantic attractions.
# ------------------------------------------------------------
nodes_with_semantics = (
    node_semantic_status[
        "semantic_missing"
    ]
    .eq(0)
    .sum()
)


# ------------------------------------------------------------
# Number of nodes without semantic attractions.
# ------------------------------------------------------------
nodes_without_semantics = (
    node_semantic_status[
        "semantic_missing"
    ]
    .eq(1)
    .sum()
)


print(
    "\nTotal graph nodes:",
    len(node_semantic_status)
)

print(
    "Nodes with attractions:",
    nodes_with_semantics
)

print(
    "Nodes without attractions:",
    nodes_without_semantics
)


# ------------------------------------------------------------
# Summarize how many attractions occur within CTs that have
# at least one attraction.
# ------------------------------------------------------------
print(
    "\nAttraction-count distribution:"
)

print(
    node_semantic_status[
        "semantic_attraction_count"
    ]
    .describe()
)


# ------------------------------------------------------------
# Show CTs with the largest number of semantic attractions.
# This helps identify whether a small number of central CTs
# contain unusually many places.
# ------------------------------------------------------------
print(
    "\nTop 20 CTs by attraction count:"
)

display(
    node_semantic_status
    .sort_values(
        "semantic_attraction_count",
        ascending=False,
    )
    .head(20)
)

Attractions retained after spatial filtering: 4094
Duplicated attraction IDs after spatial join: 0

Total graph nodes: 1248
Nodes with attractions: 623
Nodes without attractions: 625

Attraction-count distribution:
count    1248.000000
mean        3.280449
std        10.482193
min         0.000000
25%         0.000000
50%         0.000000
75%         3.000000
max       228.000000
Name: semantic_attraction_count, dtype: float64

Top 20 CTs by attraction count:


,loc_id,loc_name,semantic_attraction_count,semantic_missing
82,5350061.00,0061.00,228,0
84,5350062.03,0062.03,111,0
51,5350035.00,0035.00,107,0
27,5350014.00,0014.00,103,0
427,5350311.06,0311.06,76,0
15,5350008.02,0008.02,68,0
50,5350034.02,0034.02,68,0
28,5350015.00,0015.00,54,0
20,5350011.02,0011.02,45,0
7,5350002.00,0002.00,44,0


In [73]:
# ============================================================
# Cell 4 Ã¢â‚¬â€ Remove attraction identity from semantic text
#
# Purpose:
# Remove explicit attraction/place names from semantic_text
# before MiniLM/GPT embedding.
#
# Why:
# Wikipedia summaries commonly begin with the entity name:
#
#     "High Park is a municipal park..."
#
# We want:
#
#     "is a municipal park..."
#
# This reduces entity-specific information and makes the
# semantic representation focus more on characteristics,
# functions and context.
#
# We remove:
# 1. place_name
# 2. Wikipedia page title, when available
#
# Input:
#     attractions_assigned
#
# Output:
#     attractions_assigned["semantic_text_clean"]
# ============================================================

import re
import html
import pandas as pd
from urllib.parse import unquote


def extract_wikipedia_title_from_url(url):
    """
    Extract a readable Wikipedia page title from its URL.

    Example:
        https://en.wikipedia.org/wiki/High_Park

    becomes:
        High Park
    """

    # --------------------------------------------------------
    # Return None when the Wikipedia URL is missing.
    # --------------------------------------------------------
    if pd.isna(url):
        return None

    # --------------------------------------------------------
    # Extract everything after '/wiki/'.
    # --------------------------------------------------------
    title = str(url).split("/wiki/")[-1]

    # --------------------------------------------------------
    # Decode URL-encoded characters.
    #
    # Example:
    # %C3%89cole -> Ãƒâ€°cole
    # --------------------------------------------------------
    title = unquote(title)

    # --------------------------------------------------------
    # Wikipedia URLs use underscores instead of spaces.
    # --------------------------------------------------------
    title = title.replace("_", " ")

    return title.strip()


def clean_semantic_text_without_identity(row):
    """
    Clean one attraction's semantic text while removing
    explicit entity identity.

    The function preserves normal natural language because
    MiniLM/GPT benefit from sentence structure.
    """

    # --------------------------------------------------------
    # Get the selected semantic text.
    # --------------------------------------------------------
    text = row["semantic_text"]

    # --------------------------------------------------------
    # Missing semantic text should remain missing.
    # --------------------------------------------------------
    if pd.isna(text):
        return None

    # --------------------------------------------------------
    # Convert to string and decode any HTML entities.
    # --------------------------------------------------------
    text = html.unescape(
        str(text)
    )

    # --------------------------------------------------------
    # Remove HTML tags if any survived the Wikipedia retrieval.
    # --------------------------------------------------------
    text = re.sub(
        r"<[^>]+>",
        " ",
        text
    )

    # --------------------------------------------------------
    # Remove Wikipedia-style numeric citation markers.
    #
    # Examples:
    # [1]
    # [12]
    # --------------------------------------------------------
    text = re.sub(
        r"\[\d+\]",
        " ",
        text
    )

    # --------------------------------------------------------
    # Remove raw URLs if any appear inside the summary.
    # --------------------------------------------------------
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    # --------------------------------------------------------
    # Build a list of entity names that should be removed.
    #
    # We use both:
    # - place_name
    # - Wikipedia page title
    #
    # because one of them may differ slightly from the other.
    # --------------------------------------------------------
    names_to_remove = []

    # --------------------------------------------------------
    # Add place_name when it is a real readable name.
    #
    # Do not add Q-IDs such as Q3578367 because those do not
    # appear naturally in the Wikipedia summary.
    # --------------------------------------------------------
    place_name = row.get(
        "place_name"
    )

    if pd.notna(place_name):

        place_name = str(
            place_name
        ).strip()

        # Keep only names that are not Wikidata Q-IDs.
        if not re.fullmatch(
            r"Q\d+",
            place_name
        ):
            names_to_remove.append(
                place_name
            )

    # --------------------------------------------------------
    # Extract and add the Wikipedia page title.
    # --------------------------------------------------------
    wikipedia_title = (
        extract_wikipedia_title_from_url(
            row.get("wikipedia_url")
        )
    )

    if wikipedia_title:
        names_to_remove.append(
            wikipedia_title
        )

    # --------------------------------------------------------
    # Remove each exact entity name case-insensitively.
    #
    # re.escape() ensures punctuation in names is treated
    # literally rather than as regular-expression syntax.
    # --------------------------------------------------------
    for name in names_to_remove:

        text = re.sub(
            re.escape(name),
            " ",
            text,
            flags=re.IGNORECASE
        )

    # --------------------------------------------------------
    # Collapse repeated whitespace introduced by cleaning.
    # --------------------------------------------------------
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    # --------------------------------------------------------
    # Remove awkward spaces before punctuation.
    # --------------------------------------------------------
    text = re.sub(
        r"\s+([,.!?;:])",
        r"\1",
        text
    )

    # --------------------------------------------------------
    # Clean leading/trailing whitespace and punctuation.
    # --------------------------------------------------------
    text = text.strip(
        " ,;:-"
    )

    # --------------------------------------------------------
    # Treat an empty result as missing.
    # --------------------------------------------------------
    if text == "":
        return None

    return text


# ------------------------------------------------------------
# Apply identity removal to every attraction that belongs
# to a graph CT.
# ------------------------------------------------------------
attractions_assigned[
    "semantic_text_clean"
] = (
    attractions_assigned
    .apply(
        clean_semantic_text_without_identity,
        axis=1,
    )
)


# ============================================================
# VALIDATION
# ============================================================


# ------------------------------------------------------------
# Count missing texts after cleaning.
#
# Ideally this should be 0.
# ------------------------------------------------------------
print(
    "Attractions:",
    len(attractions_assigned)
)

print(
    "Missing semantic_text_clean:",
    attractions_assigned[
        "semantic_text_clean"
    ]
    .isna()
    .sum()
)


# ------------------------------------------------------------
# Compare original text with cleaned text for inspection.
# ------------------------------------------------------------
display(
    attractions_assigned[
        [
            "wikidata_id",
            "place_name",
            "semantic_source",
            "semantic_text",
            "semantic_text_clean",
        ]
    ]
    .head(20)
)

Attractions: 4094
Missing semantic_text_clean: 1


,wikidata_id,place_name,semantic_source,semantic_text,semantic_text_clean
0,Q3578367,Q3578367,wikidata,"public high school in Toronto, Ontario, Canada","public high school in Toronto, Ontario, Canada"
1,Q16890845,École secondaire Toronto Ouest,wikidata,"high school in Brockton Village, Toronto, Onta...","high school in Brockton Village, Toronto, Onta..."
2,Q14875502,École secondaire catholique Monseigneur-de-Cha...,type_only,high school,high school
3,Q65077953,École secondaire catholique Père-Philippe-Lama...,wikidata,public separate secondary school in Eglinton E...,public separate secondary school in Eglinton E...
4,Q16834547,École secondaire catholique Saint-Frère-André,wikidata,"high school in Brockton Village, Toronto, Onta...","high school in Brockton Village, Toronto, Onta..."
5,Q14875147,Etienne Brule Park,wikidata,"park in Ontario, Canada","park in Ontario, Canada"
6,Q124322743,10 Armoury Street,wikipedia,"10 Armoury Street in Toronto, Ontario, Canada,...","in Toronto, Ontario, Canada, is the site of a ..."
7,Q139973819,1414 Danforth Avenue,wikipedia,1414 Danforth Avenue (2nd floor offices munici...,(2nd floor offices municipally referred to as ...
8,Q4569309,1958 Jim Mideon 500,wikipedia,The 1958 Jim Mideon 500 (known officially as 1...,The (known officially as 1958-31) was a NASCAR...
9,Q139855956,1975 Scarborough bus-train collision,wikipedia,The 1975 Scarborough bus-train collision occur...,The occurred on December 12th 1975 when a Toro...


In [74]:
attractions_assigned.columns

Index(['wikipedia_url', 'wikidata_id', 'place_name', 'place_type',
       'wikidata_description', 'coord', 'wikipedia_title', 'wikipedia_summary',
       'semantic_text', 'semantic_source', 'geometry', 'attr_lon', 'attr_lat',
       'loc_id', 'loc_name', 'node_lon', 'node_lat', 'node_point_geom',
       'semantic_text_clean'],
      dtype='str')

In [75]:
# ============================================================
# Cell 4B Ã¢â‚¬â€ Repair missing semantic_text_clean values
#
# Problem:
# Some semantic texts became empty after removing the explicit
# attraction/place name.
#
# Solution:
#
# 1. Clean Wikipedia summary and remove attraction identity.
# 2. If nothing remains, try Wikidata description.
# 3. If nothing remains, use place_type.
# 4. If all three are unusable, drop the attraction.
#
# This preserves the SAME fallback logic used previously,
# but applies it AFTER identity removal.
# ============================================================

import re
import html
import pandas as pd
from urllib.parse import unquote


# ------------------------------------------------------------
# First inspect how many attraction texts are currently missing.
# ------------------------------------------------------------
missing_before = (
    attractions_assigned["semantic_text_clean"]
    .isna()
    .sum()
)

print(
    "Missing semantic_text_clean before repair:",
    missing_before
)


# ------------------------------------------------------------
# Display the problematic attractions before changing anything.
# ------------------------------------------------------------
display(
    attractions_assigned.loc[
        attractions_assigned["semantic_text_clean"].isna(),
        [
            "wikidata_id",
            "place_name",
            "place_type",
            "wikidata_description",
            "wikipedia_url",
            "wikipedia_summary",
            "semantic_source",
            "semantic_text",
        ]
    ]
)

Missing semantic_text_clean before repair: 1


,wikidata_id,place_name,place_type,wikidata_description,wikipedia_url,wikipedia_summary,semantic_source,semantic_text
3741,Q5029793,Canadian Business College,business school,Canadian business college,NaN,NaN,wikidata,Canadian business college


In [76]:
# ============================================================
# Check whether the repair has actually fixed all missing text
# ============================================================

# ------------------------------------------------------------
# Count missing semantic texts in the CURRENT dataframe.
# ------------------------------------------------------------
missing_count = (
    attractions_assigned[
        "semantic_text_clean"
    ]
    .isna()
    .sum()
)

print(
    "Missing semantic_text_clean:",
    missing_count
)

Missing semantic_text_clean: 1


In [77]:
# ============================================================
# Cell 4C Ã¢â‚¬â€ Repair remaining missing semantic_text_clean
#
# Fallback hierarchy AFTER removing place identity:
#
# 1. Wikipedia summary
# 2. Wikidata description
# 3. place_type
# 4. Drop only if all are unavailable
#
# This cell only operates on rows where
# semantic_text_clean is currently missing.
# ============================================================

import re
import pandas as pd


def remove_place_name_from_text(text, place_name):
    """
    Remove the explicit attraction/place name from a text.

    Returns:
        cleaned text, or None if nothing useful remains.
    """

    # --------------------------------------------------------
    # Missing source text cannot be used.
    # --------------------------------------------------------
    if pd.isna(text):
        return None

    # --------------------------------------------------------
    # Convert source text to a clean string.
    # --------------------------------------------------------
    text = str(text).strip()

    # --------------------------------------------------------
    # Remove the place name when it exists and is not a Q-ID.
    # --------------------------------------------------------
    if pd.notna(place_name):

        place_name = str(place_name).strip()

        # Ignore values such as Q123456 because these are not
        # meaningful natural-language place names.
        if not re.fullmatch(r"Q\d+", place_name):

            text = re.sub(
                re.escape(place_name),
                " ",
                text,
                flags=re.IGNORECASE,
            )

    # --------------------------------------------------------
    # Collapse repeated whitespace caused by name removal.
    # --------------------------------------------------------
    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip(" ,;:-")

    # --------------------------------------------------------
    # Return None if removing the place name leaves nothing.
    # --------------------------------------------------------
    if text == "":
        return None

    return text


def repair_missing_semantic_text(row):
    """
    Repair a row using the agreed semantic fallback:

        Wikipedia summary
            Ã¢â€ â€œ
        Wikidata description
            Ã¢â€ â€œ
        place_type
            Ã¢â€ â€œ
        None
    """

    # --------------------------------------------------------
    # 1. Try Wikipedia summary first.
    # --------------------------------------------------------
    wiki_text = remove_place_name_from_text(
        row["wikipedia_summary"],
        row["place_name"],
    )

    if wiki_text is not None:

        return pd.Series({
            "semantic_text_clean": wiki_text,
            "semantic_source_clean": "wikipedia",
        })

    # --------------------------------------------------------
    # 2. Try Wikidata description.
    # --------------------------------------------------------
    wikidata_text = remove_place_name_from_text(
        row["wikidata_description"],
        row["place_name"],
    )

    if wikidata_text is not None:

        return pd.Series({
            "semantic_text_clean": wikidata_text,
            "semantic_source_clean": "wikidata",
        })

    # --------------------------------------------------------
    # 3. Fall back to place_type.
    #
    # For Canadian Business College this should produce:
    #
    #     business school
    # --------------------------------------------------------
    if pd.notna(row["place_type"]):

        place_type = str(
            row["place_type"]
        ).strip()

        if place_type != "":

            return pd.Series({
                "semantic_text_clean": place_type,
                "semantic_source_clean": "type_only",
            })

    # --------------------------------------------------------
    # 4. No usable semantic information exists.
    # --------------------------------------------------------
    return pd.Series({
        "semantic_text_clean": None,
        "semantic_source_clean": "none",
    })


# ------------------------------------------------------------
# Find only the currently missing rows.
# ------------------------------------------------------------
missing_mask = (
    attractions_assigned[
        "semantic_text_clean"
    ]
    .isna()
)


# ------------------------------------------------------------
# Repair ONLY those missing rows.
# ------------------------------------------------------------
repair_result = (
    attractions_assigned.loc[
        missing_mask
    ]
    .apply(
        repair_missing_semantic_text,
        axis=1,
    )
)


# ------------------------------------------------------------
# Write the repaired values back into attractions_assigned.
# ------------------------------------------------------------
attractions_assigned.loc[
    missing_mask,
    [
        "semantic_text_clean",
        "semantic_source_clean",
    ]
] = repair_result[
    [
        "semantic_text_clean",
        "semantic_source_clean",
    ]
].values


# ============================================================
# VALIDATION
# ============================================================

# ------------------------------------------------------------
# This should now be 0.
# ------------------------------------------------------------
print(
    "Missing semantic_text_clean after repair:",
    attractions_assigned[
        "semantic_text_clean"
    ]
    .isna()
    .sum()
)


# ------------------------------------------------------------
# Inspect the repaired row.
# ------------------------------------------------------------
display(
    attractions_assigned.loc[
        missing_mask,
        [
            "wikidata_id",
            "place_name",
            "place_type",
            "wikidata_description",
            "semantic_source_clean",
            "semantic_text_clean",
        ]
    ]
)

Missing semantic_text_clean after repair: 0


,wikidata_id,place_name,place_type,wikidata_description,semantic_source_clean,semantic_text_clean
3741,Q5029793,Canadian Business College,business school,Canadian business college,type_only,business school


In [78]:
### write the data

In [79]:
# ============================================================
# Cell 5 Ã¢â‚¬â€ Embed each attraction and keep spatial identifiers
#
# Purpose:
# 1. Embed each attraction independently using MiniLM.
# 2. Keep each embedding linked to:
#       - wikidata_id
#       - place_name
#       - loc_id
#       - loc_name
#       - attr_lon
#       - attr_lat
#       - node_lon
#       - node_lat
#
# This ensures every attraction embedding remains spatially
# traceable before aggregation to CT/node level.
#
# Input:
#     attractions_assigned
#
# Output:
#     attraction_embedding_df
#
# One row = one attraction
# ============================================================

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer


# ------------------------------------------------------------
# Select GPU when available.
# ------------------------------------------------------------
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Embedding device:",
    device
)


# ------------------------------------------------------------
# Load MiniLM.
#
# Output dimension:
#     384
# ------------------------------------------------------------
minilm_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device,
)


# ------------------------------------------------------------
# Confirm that every assigned attraction has semantic text.
# ------------------------------------------------------------
assert (
    attractions_assigned[
        "semantic_text_clean"
    ]
    .notna()
    .all()
), "Some attractions still have missing semantic text."


# ------------------------------------------------------------
# Extract the attraction texts in their CURRENT row order.
#
# This row order must remain unchanged because the embedding
# matrix will correspond row-for-row to attractions_assigned.
# ------------------------------------------------------------
attraction_texts = (
    attractions_assigned[
        "semantic_text_clean"
    ]
    .tolist()
)


# ------------------------------------------------------------
# Generate one embedding for every attraction.
#
# Shape expected:
#     (4119, 384)
# ------------------------------------------------------------
attraction_embeddings = (
    minilm_model.encode(
        attraction_texts,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
)


# ------------------------------------------------------------
# Validate embedding dimensions.
# ------------------------------------------------------------
assert (
    attraction_embeddings.shape[0]
    == len(attractions_assigned)
), "Embedding rows do not match attraction rows."

assert (
    attraction_embeddings.shape[1]
    == 384
), "Unexpected MiniLM embedding dimension."


# ------------------------------------------------------------
# Create names for the 384 semantic dimensions.
#
# Example:
#     minilm_000
#     minilm_001
#     ...
#     minilm_383
# ------------------------------------------------------------
embedding_cols = [
    f"minilm_{i:03d}"
    for i in range(
        attraction_embeddings.shape[1]
    )
]


# ------------------------------------------------------------
# Convert the embedding matrix into a DataFrame.
#
# IMPORTANT:
# Use the SAME index as attractions_assigned so the vectors
# stay aligned with the correct attraction rows.
# ------------------------------------------------------------
embedding_features = pd.DataFrame(
    attraction_embeddings,
    columns=embedding_cols,
    index=attractions_assigned.index,
)


# ------------------------------------------------------------
# Keep the spatial and identification fields needed later.
#
# place_name and wikidata_id are retained for auditing.
# They are NOT inputs to MiniLM.
# ------------------------------------------------------------
attraction_embedding_df = (
    attractions_assigned[
        [
            "wikidata_id",
            "place_name",
            "loc_id",
            "loc_name",
            "attr_lon",
            "attr_lat",
            "node_lon",
            "node_lat",
            "semantic_source",
            "semantic_text_clean",
        ]
    ]
    .copy()
)



# ------------------------------------------------------------
# Attach the 384-dimensional embedding to each attraction row.
# ------------------------------------------------------------
attraction_embedding_df = pd.concat(
    [
        attraction_embedding_df,
        embedding_features,
    ],
    axis=1,
)


# ============================================================
# VALIDATION
# ============================================================

print(
    "\nAttraction-level embedding table shape:",
    attraction_embedding_df.shape
)

print(
    "Attractions:",
    len(attraction_embedding_df)
)

print(
    "Unique CTs represented:",
    attraction_embedding_df[
        "loc_id"
    ].nunique()
)

print(
    "Missing loc_id:",
    attraction_embedding_df[
        "loc_id"
    ].isna().sum()
)

print(
    "Missing attraction longitude:",
    attraction_embedding_df[
        "attr_lon"
    ].isna().sum()
)

print(
    "Missing attraction latitude:",
    attraction_embedding_df[
        "attr_lat"
    ].isna().sum()
)

print(
    "NaNs inside MiniLM embeddings:",
    attraction_embedding_df[
        embedding_cols
    ].isna().sum().sum()
)


# ------------------------------------------------------------
# Inspect the relationship:
#
# attraction
#     -> coordinates
#     -> loc_id
#     -> MiniLM vector
# ------------------------------------------------------------
display(
    attraction_embedding_df[
        [
            "wikidata_id",
            "place_name",
            "loc_id",
            "loc_name",
            "attr_lon",
            "attr_lat",
            "node_lon",
            "node_lat",
            "minilm_000",
            "minilm_001",
            "minilm_002",
        ]
    ]
    .head(10)
)

Embedding device: cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/64 [00:00<?, ?it/s]


Attraction-level embedding table shape: (4094, 394)
Attractions: 4094
Unique CTs represented: 623
Missing loc_id: 0
Missing attraction longitude: 0
Missing attraction latitude: 0
NaNs inside MiniLM embeddings: 0


,wikidata_id,place_name,loc_id,loc_name,attr_lon,attr_lat,node_lon,node_lat,minilm_000,minilm_001,minilm_002
0,Q3578367,Q3578367,5350273.02,0273.02,-79.371500,43.752200,-79.373296,43.753904,0.066409,0.031791,0.024167
1,Q16890845,École secondaire Toronto Ouest,5350053.00,0053.00,-79.441000,43.652200,-79.441967,43.654413,0.007707,0.005757,0.014058
2,Q14875502,École secondaire catholique Monseigneur-de-Cha...,5350319.00,0319.00,-79.423000,43.786000,-79.427736,43.788277,-0.039380,0.068518,-0.018138
3,Q65077953,École secondaire catholique Père-Philippe-Lama...,5350355.04,0355.04,-79.240707,43.738784,-79.242831,43.736569,0.065201,-0.008228,0.025973
4,Q16834547,École secondaire catholique Saint-Frère-André,5350053.00,0053.00,-79.441000,43.652200,-79.441967,43.654413,0.007707,0.005757,0.014058
5,Q14875147,Etienne Brule Park,5350150.00,0150.00,-79.494400,43.652800,-79.492909,43.654389,0.085981,0.007590,0.067141
6,Q124322743,10 Armoury Street,5350035.00,0035.00,-79.386100,43.654100,-79.384952,43.656109,0.033516,0.016777,-0.023485
7,Q139973819,1414 Danforth Avenue,5350082.00,0082.00,-79.327400,43.682710,-79.328752,43.684222,0.010626,0.022897,0.029945
8,Q4569309,1958 Jim Mideon 500,5350008.02,0008.02,-79.417778,43.631944,-79.412563,43.633826,0.018085,0.084746,0.030577
9,Q139855956,1975 Scarborough bus-train collision,5350343.00,0343.00,-79.254300,43.718000,-79.263601,43.711094,0.084692,0.001663,0.032447


In [80]:
# ============================================================
# Cell 6 Ã¢â‚¬â€ Aggregate attraction embeddings to CT/node level
#
# Purpose:
#
# Each attraction already has:
#     loc_id
#     384-dimensional MiniLM embedding
#
# We now:
#
# 1. Mean-pool attraction embeddings within each loc_id.
# 2. Count attractions within each loc_id.
# 3. Start from ALL 1,248 graph nodes.
# 4. CTs without attractions receive:
#       - zero semantic vector
#       - attraction_count = 0
#       - attraction_missing_flag = 1
#
# Final output:
#     node_wiki_embeddings
#
# Final columns:
#     loc_id
#     wiki_emb_0
#     wiki_emb_1
#     ...
#     wiki_emb_383
#     attraction_count
#     attraction_missing_flag
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# Identify the 384 attraction-level MiniLM embedding columns.
#
# Current names:
#     minilm_000
#     minilm_001
#     ...
#     minilm_383
# ------------------------------------------------------------
minilm_cols = [
    f"minilm_{i:03d}"
    for i in range(384)
]


# ------------------------------------------------------------
# Validate that all expected MiniLM columns exist before
# aggregation.
# ------------------------------------------------------------
missing_embedding_cols = [
    col
    for col in minilm_cols
    if col not in attraction_embedding_df.columns
]

assert (
    len(missing_embedding_cols) == 0
), (
    "Some MiniLM embedding columns are missing: "
    f"{missing_embedding_cols}"
)


# ============================================================
# STEP 1 Ã¢â‚¬â€ Mean-pool attraction embeddings by loc_id
# ============================================================

# ------------------------------------------------------------
# For each CT, calculate the mean of all attraction embeddings.
#
# Example:
#
# CT A:
#     attraction 1 -> vector_1
#     attraction 2 -> vector_2
#     attraction 3 -> vector_3
#
# becomes:
#
#     CT_A_embedding =
#         mean(vector_1, vector_2, vector_3)
#
# Result:
#     one 384-dimensional semantic vector per CT that has
#     at least one attraction.
# ------------------------------------------------------------
ct_mean_embeddings = (
    attraction_embedding_df
    .groupby(
        "loc_id",
        as_index=False
    )[minilm_cols]
    .mean()
)


# ============================================================
# STEP 2 Ã¢â‚¬â€ Rename embedding columns
# ============================================================

# ------------------------------------------------------------
# Rename:
#
#     minilm_000 -> wiki_emb_0
#     minilm_001 -> wiki_emb_1
#     ...
#     minilm_383 -> wiki_emb_383
#
# These are the final semantic feature names.
# ------------------------------------------------------------
embedding_rename_map = {
    f"minilm_{i:03d}": f"wiki_emb_{i}"
    for i in range(384)
}

ct_mean_embeddings = (
    ct_mean_embeddings
    .rename(
        columns=embedding_rename_map
    )
)


# ------------------------------------------------------------
# Store the final embedding-column names for later operations.
# ------------------------------------------------------------
wiki_embedding_cols = [
    f"wiki_emb_{i}"
    for i in range(384)
]


# ============================================================
# STEP 3 Ã¢â‚¬â€ Count attractions per CT
# ============================================================

# ------------------------------------------------------------
# Count the number of attraction records assigned to each CT.
#
# This remains separate from the mean embedding because:
#
#     embedding -> semantic composition
#     count     -> quantity of attractions
# ------------------------------------------------------------
ct_attraction_counts = (
    attraction_embedding_df
    .groupby("loc_id")
    .size()
    .reset_index(
        name="attraction_count"
    )
)


# ============================================================
# STEP 4 Ã¢â‚¬â€ Start from ALL graph nodes
# ============================================================

# ------------------------------------------------------------
# Start from nodes rather than from attraction data.
#
# This ensures that CTs with ZERO attractions are retained.
#
# Expected:
#     1,248 rows
# ------------------------------------------------------------
node_wiki_embeddings = (
    nodes[
        ["loc_id"]
    ]
    .drop_duplicates()
    .copy()
)


# ============================================================
# STEP 5 Ã¢â‚¬â€ Join CT mean embeddings
# ============================================================

# ------------------------------------------------------------
# Left join means:
#
# CT with attractions:
#     receives its mean embedding.
#
# CT without attractions:
#     embedding columns temporarily become NaN.
# ------------------------------------------------------------
node_wiki_embeddings = (
    node_wiki_embeddings
    .merge(
        ct_mean_embeddings,
        on="loc_id",
        how="left"
    )
)


# ============================================================
# STEP 6 Ã¢â‚¬â€ Join attraction counts
# ============================================================

# ------------------------------------------------------------
# Attach the number of attractions assigned to each CT.
#
# CTs without attractions temporarily receive NaN.
# ------------------------------------------------------------
node_wiki_embeddings = (
    node_wiki_embeddings
    .merge(
        ct_attraction_counts,
        on="loc_id",
        how="left"
    )
)


# ============================================================
# STEP 7 Ã¢â‚¬â€ Handle CTs without attractions
# ============================================================

# ------------------------------------------------------------
# Missing attraction_count means the CT had no attraction.
#
# Use 0 because this is a genuine count variable.
# ------------------------------------------------------------
node_wiki_embeddings[
    "attraction_count"
] = (
    node_wiki_embeddings[
        "attraction_count"
    ]
    .fillna(0)
    .astype(int)
)


# ------------------------------------------------------------
# Create the explicit missing-attraction flag.
#
# 0 = one or more attractions available
# 1 = no attractions available
# ------------------------------------------------------------
node_wiki_embeddings[
    "attraction_missing_flag"
] = (
    node_wiki_embeddings[
        "attraction_count"
    ]
    .eq(0)
    .astype(int)
)


# ------------------------------------------------------------
# CTs without attractions currently have NaN in all 384
# semantic dimensions.
#
# Replace those NaNs with a zero vector.
#
# The attraction_missing_flag distinguishes:
#
#     real semantic embedding
#
# from:
#
#     zero vector representing no attraction information.
# ------------------------------------------------------------
node_wiki_embeddings[
    wiki_embedding_cols
] = (
    node_wiki_embeddings[
        wiki_embedding_cols
    ]
    .fillna(0.0)
)


# ============================================================
# STEP 8 Ã¢â‚¬â€ Put columns in the exact requested order
# ============================================================

# ------------------------------------------------------------
# Final structure:
#
# loc_id
# wiki_emb_0
# ...
# wiki_emb_383
# attraction_count
# attraction_missing_flag
# ------------------------------------------------------------
final_columns = (
    ["loc_id"]
    + wiki_embedding_cols
    + [
        "attraction_count",
        "attraction_missing_flag",
    ]
)

node_wiki_embeddings = (
    node_wiki_embeddings[
        final_columns
    ]
    .copy()
)


# ============================================================
# FINAL VALIDATION
# ============================================================

# ------------------------------------------------------------
# Expected final shape:
#
# 1248 rows
# 387 columns
#
# 1 loc_id
# + 384 embeddings
# + 1 attraction_count
# + 1 attraction_missing_flag
# = 387
# ------------------------------------------------------------
print(
    "Final table shape:",
    node_wiki_embeddings.shape
)


# ------------------------------------------------------------
# Verify that all graph nodes are represented exactly once.
# ------------------------------------------------------------
print(
    "Unique loc_id:",
    node_wiki_embeddings[
        "loc_id"
    ].nunique()
)


print(
    "Duplicated loc_id:",
    node_wiki_embeddings[
        "loc_id"
    ].duplicated().sum()
)


# ------------------------------------------------------------
# Verify attraction totals.
#
# Expected:
#     4119
# ------------------------------------------------------------
print(
    "Total attraction_count:",
    node_wiki_embeddings[
        "attraction_count"
    ].sum()
)


# ------------------------------------------------------------
# Expected from your previous result:
#
# nodes with attractions    = 625
# nodes without attractions = 623
# ------------------------------------------------------------
print(
    "Nodes with attractions:",
    (
        node_wiki_embeddings[
            "attraction_missing_flag"
        ] == 0
    ).sum()
)


print(
    "Nodes without attractions:",
    (
        node_wiki_embeddings[
            "attraction_missing_flag"
        ] == 1
    ).sum()
)


# ------------------------------------------------------------
# There should be no NaNs anywhere in the semantic embeddings.
# ------------------------------------------------------------
print(
    "NaNs in wiki embeddings:",
    node_wiki_embeddings[
        wiki_embedding_cols
    ]
    .isna()
    .sum()
    .sum()
)


# ------------------------------------------------------------
# Verify that CTs marked as missing have exactly zero vectors.
# ------------------------------------------------------------
missing_nodes = (
    node_wiki_embeddings[
        "attraction_missing_flag"
    ] == 1
)

nonzero_values_in_missing_nodes = (
    node_wiki_embeddings.loc[
        missing_nodes,
        wiki_embedding_cols
    ]
    .to_numpy()
    != 0
).sum()

print(
    "Non-zero embedding values in missing CTs:",
    nonzero_values_in_missing_nodes
)


# ------------------------------------------------------------
# Inspect the final table.
# ------------------------------------------------------------
display(
    node_wiki_embeddings[
        [
            "loc_id",
            "wiki_emb_0",
            "wiki_emb_1",
            "wiki_emb_2",
            "wiki_emb_3",
            "attraction_count",
            "attraction_missing_flag",
        ]
    ]
    .head(20)
)

Final table shape: (1248, 387)
Unique loc_id: 1248
Duplicated loc_id: 0
Total attraction_count: 4094
Nodes with attractions: 623
Nodes without attractions: 625
NaNs in wiki embeddings: 0
Non-zero embedding values in missing CTs: 0


,loc_id,wiki_emb_0,wiki_emb_1,wiki_emb_2,wiki_emb_3,attraction_count,attraction_missing_flag
0,5320100.01,0.000000,0.000000,0.000000,0.000000,0,1
1,5320100.02,0.000000,0.000000,0.000000,0.000000,0,1
2,5320100.03,0.000000,0.000000,0.000000,0.000000,0,1
3,5320105.14,0.000000,0.000000,0.000000,0.000000,0,1
4,5320105.17,0.000000,0.000000,0.000000,0.000000,0,1
5,5320105.22,0.000000,0.000000,0.000000,0.000000,0,1
6,5350001.00,0.038500,0.007291,0.019660,-0.008674,29,0
7,5350002.00,0.049830,0.008827,0.015737,-0.006718,44,0
8,5350003.00,0.054344,0.013804,0.013618,-0.014471,12,0
9,5350004.00,0.040659,0.013924,0.025616,-0.001248,5,0


In [81]:
# Save the final node-level MiniLM semantic embedding table as Parquet.
output_path = (
    project_root
    / "data/processed/graph/desc_semantic/desc_all_wikidata_minilm_embeddings.parquet"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

node_wiki_embeddings.to_parquet(
    output_path,
    index=False,
)

print("Saved MiniLM node embeddings:", output_path)
print("Shape:", node_wiki_embeddings.shape)
print("Total attraction_count:", node_wiki_embeddings["attraction_count"].sum())
print("Nodes without attractions:", node_wiki_embeddings["attraction_missing_flag"].sum())


Saved MiniLM node embeddings: /home/najla/dev/najla-msc/bikeshare/data/processed/graph/desc_semantic/desc_all_wikidata_minilm_embeddings.parquet
Shape: (1248, 387)
Total attraction_count: 4094
Nodes without attractions: 625


GPT

In [82]:
# ============================================================
# Cell 7 - GPT embeddings for attraction descriptions
#
# Multiple descriptions per node are handled the same way as
# the MiniLM pipeline above:
#
# 1. Embed each attraction description independently.
# 2. Mean-pool attraction embeddings within each loc_id.
# 3. Keep attraction_count as a separate quantity feature.
# 4. Give nodes with no attractions a zero vector plus an
#    attraction_missing_flag = 1.
#
# Naming convention:
# - GPT large columns: gpt_emb_0, gpt_emb_1, ...
# - GPT small columns: gpt_small_emb_0, gpt_small_emb_1, ...
# - MiniLM columns remain wiki_emb_0, wiki_emb_1, ...
# ============================================================

import os
from getpass import getpass

import numpy as np
import pandas as pd
from openai import OpenAI


# ------------------------------------------------------------
# Configure OpenAI client.
# ------------------------------------------------------------
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Paste OpenAI API key:")

client = OpenAI()

GPT_EMBEDDING_MODEL_LARGE = "text-embedding-3-large"
GPT_EMBEDDING_MODEL_SMALL = "text-embedding-3-small"


def l2_normalize_rows(matrix, eps=1e-12):
    # Normalize each embedding row before node-level pooling.
    norms = np.linalg.norm(
        matrix,
        axis=1,
        keepdims=True,
    )

    return matrix / np.maximum(norms, eps)


def create_gpt_embeddings(text_list, model_name, batch_size=100):
    # Create OpenAI embeddings in batches and preserve input order.
    all_embeddings = []

    for start in range(0, len(text_list), batch_size):
        batch_text = text_list[start:start + batch_size]

        response = client.embeddings.create(
            model=model_name,
            input=batch_text,
        )

        batch_embeddings = [
            item.embedding
            for item in sorted(response.data, key=lambda item: item.index)
        ]

        all_embeddings.extend(batch_embeddings)

        print(
            f"{model_name}: embedded "
            f"{min(start + batch_size, len(text_list))} / {len(text_list)}"
        )

    return np.asarray(
        all_embeddings,
        dtype=np.float32,
    )


def build_gpt_node_embedding_table(
    model_name,
    output_path,
    embedding_prefix,
    batch_size=100,
):
    # Embed attraction descriptions, then aggregate to one row per graph node.
    assert (
        attractions_assigned["semantic_text_clean"]
        .notna()
        .all()
    ), "Some attractions still have missing semantic_text_clean."

    attraction_texts = (
        attractions_assigned["semantic_text_clean"]
        .astype(str)
        .tolist()
    )

    attraction_embeddings = create_gpt_embeddings(
        text_list=attraction_texts,
        model_name=model_name,
        batch_size=batch_size,
    )

    # Match the MiniLM treatment: pool normalized attraction vectors.
    attraction_embeddings = l2_normalize_rows(
        attraction_embeddings
    )

    raw_embedding_cols = [
        f"gpt_raw_{i:04d}"
        for i in range(attraction_embeddings.shape[1])
    ]

    embedding_features = pd.DataFrame(
        attraction_embeddings,
        columns=raw_embedding_cols,
        index=attractions_assigned.index,
    )

    attraction_gpt_embedding_df = pd.concat(
        [
            attractions_assigned[
                [
                    "wikidata_id",
                    "place_name",
                    "loc_id",
                    "loc_name",
                    "attr_lon",
                    "attr_lat",
                    "node_lon",
                    "node_lat",
                    "semantic_source",
                    "semantic_text_clean",
                ]
            ].copy(),
            embedding_features,
        ],
        axis=1,
    )

    ct_mean_embeddings = (
        attraction_gpt_embedding_df
        .groupby("loc_id", as_index=False)[raw_embedding_cols]
        .mean()
    )

    embedding_cols = [
        f"{embedding_prefix}{i}"
        for i in range(len(raw_embedding_cols))
    ]

    ct_mean_embeddings = ct_mean_embeddings.rename(
        columns=dict(
            zip(
                raw_embedding_cols,
                embedding_cols,
            )
        )
    )

    ct_attraction_counts = (
        attraction_gpt_embedding_df
        .groupby("loc_id")
        .size()
        .reset_index(name="attraction_count")
    )

    node_gpt_embeddings = (
        nodes[["loc_id"]]
        .drop_duplicates()
        .copy()
        .merge(
            ct_mean_embeddings,
            on="loc_id",
            how="left",
        )
        .merge(
            ct_attraction_counts,
            on="loc_id",
            how="left",
        )
    )

    node_gpt_embeddings["attraction_count"] = (
        node_gpt_embeddings["attraction_count"]
        .fillna(0)
        .astype(int)
    )

    node_gpt_embeddings["attraction_missing_flag"] = (
        node_gpt_embeddings["attraction_count"]
        .eq(0)
        .astype(int)
    )

    node_gpt_embeddings[embedding_cols] = (
        node_gpt_embeddings[embedding_cols]
        .fillna(0.0)
    )

    if "node_wiki_embeddings" in globals():
        expected_metadata = node_wiki_embeddings[
            [
                "loc_id",
                "attraction_count",
                "attraction_missing_flag",
            ]
        ].copy()

        metadata_check = node_gpt_embeddings[
            [
                "loc_id",
                "attraction_count",
                "attraction_missing_flag",
            ]
        ].merge(
            expected_metadata,
            on="loc_id",
            how="left",
            suffixes=("_gpt", "_minilm"),
        )

        mismatched_metadata = metadata_check[
            metadata_check["attraction_count_gpt"].ne(
                metadata_check["attraction_count_minilm"]
            )
            |
            metadata_check["attraction_missing_flag_gpt"].ne(
                metadata_check["attraction_missing_flag_minilm"]
            )
        ]

        if len(mismatched_metadata) > 0:
            raise ValueError(
                "GPT node coverage does not match MiniLM node coverage. "
                f"Examples: {mismatched_metadata.head(10).to_dict(orient='records')}"
            )

    described_zero_nodes = node_gpt_embeddings.loc[
        node_gpt_embeddings["attraction_missing_flag"].eq(0)
        &
        node_gpt_embeddings[embedding_cols].abs().sum(axis=1).eq(0),
        "loc_id",
    ].tolist()

    if described_zero_nodes:
        raise ValueError(
            "GPT produced zero vectors for nodes with descriptions: "
            f"{described_zero_nodes}"
        )

    final_columns = (
        ["loc_id"]
        + embedding_cols
        + [
            "attraction_count",
            "attraction_missing_flag",
        ]
    )

    node_gpt_embeddings = node_gpt_embeddings[
        final_columns
    ].copy()

    node_gpt_embeddings.to_parquet(
        output_path,
        index=False,
    )

    print("Saved:", output_path)
    print("Final table shape:", node_gpt_embeddings.shape)
    print("Unique loc_id:", node_gpt_embeddings["loc_id"].nunique())
    print("Duplicated loc_id:", node_gpt_embeddings["loc_id"].duplicated().sum())
    print("Total attraction_count:", node_gpt_embeddings["attraction_count"].sum())
    print("Nodes without attractions:", node_gpt_embeddings["attraction_missing_flag"].sum())
    print(
        "NaNs in embeddings:",
        node_gpt_embeddings[embedding_cols].isna().sum().sum(),
    )

    return node_gpt_embeddings


gpt_large_output_path = (
    project_root
    / "data/processed/graph/desc_semantic/desc_all_wikidata_gpt_large_embeddings.parquet"
)

gpt_small_output_path = (
    project_root
    / "data/processed/graph/desc_semantic/desc_all_wikidata_gpt_small_embeddings.parquet"
)


# Run these when you are ready to spend API calls.
gpt_large_node_embeddings = build_gpt_node_embedding_table(
    model_name=GPT_EMBEDDING_MODEL_LARGE,
    output_path=gpt_large_output_path,
    embedding_prefix="gpt_emb_",
    batch_size=100,
)

gpt_small_node_embeddings = build_gpt_node_embedding_table(
    model_name=GPT_EMBEDDING_MODEL_SMALL,
    output_path=gpt_small_output_path,
    embedding_prefix="gpt_small_emb_",
    batch_size=100,
)

display(
    gpt_small_node_embeddings[
        [
            "loc_id",
            "gpt_small_emb_0",
            "gpt_small_emb_1",
            "gpt_small_emb_2",
            "attraction_count",
            "attraction_missing_flag",
        ]
    ].head(20)
)


text-embedding-3-large: embedded 100 / 4094
text-embedding-3-large: embedded 200 / 4094
text-embedding-3-large: embedded 300 / 4094
text-embedding-3-large: embedded 400 / 4094
text-embedding-3-large: embedded 500 / 4094
text-embedding-3-large: embedded 600 / 4094
text-embedding-3-large: embedded 700 / 4094
text-embedding-3-large: embedded 800 / 4094
text-embedding-3-large: embedded 900 / 4094
text-embedding-3-large: embedded 1000 / 4094
text-embedding-3-large: embedded 1100 / 4094
text-embedding-3-large: embedded 1200 / 4094
text-embedding-3-large: embedded 1300 / 4094
text-embedding-3-large: embedded 1400 / 4094
text-embedding-3-large: embedded 1500 / 4094
text-embedding-3-large: embedded 1600 / 4094
text-embedding-3-large: embedded 1700 / 4094
text-embedding-3-large: embedded 1800 / 4094
text-embedding-3-large: embedded 1900 / 4094
text-embedding-3-large: embedded 2000 / 4094
text-embedding-3-large: embedded 2100 / 4094
text-embedding-3-large: embedded 2200 / 4094
text-embedding-3-la

,loc_id,gpt_small_emb_0,gpt_small_emb_1,gpt_small_emb_2,attraction_count,attraction_missing_flag
0,5320100.01,0.000000,0.000000,0.000000,0,1
1,5320100.02,0.000000,0.000000,0.000000,0,1
2,5320100.03,0.000000,0.000000,0.000000,0,1
3,5320105.14,0.000000,0.000000,0.000000,0,1
4,5320105.17,0.000000,0.000000,0.000000,0,1
5,5320105.22,0.000000,0.000000,0.000000,0,1
6,5350001.00,-0.016359,-0.010744,0.018275,29,0
7,5350002.00,-0.029456,-0.009134,0.014067,44,0
8,5350003.00,-0.030281,-0.024264,0.011365,12,0
9,5350004.00,-0.048567,-0.008395,0.012948,5,0
